# Normalization Processing Job

Launches `src/normalize.py` as a SageMaker Processing Job.
Reads the 6 raw source TSVs from S3 and writes normalized versions to `data/normalized/`.

**Run this once.** Output files:
```
data/normalized/train/norm_s1.tsv
data/normalized/train/norm_s2.tsv
data/normalized/train/norm_s3.tsv
data/normalized/test/norm_test_s1.tsv
data/normalized/test/norm_test_s2.tsv
data/normalized/test/norm_test_s3.tsv
```

In [3]:
import boto3
import sagemaker
from sagemaker.sklearn import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

boto_session = boto3.Session(region_name='ap-southeast-2')
session      = sagemaker.Session(boto_session=boto_session)
bucket       = session.default_bucket()
role         = sagemaker.get_execution_role(sagemaker_session=session)  # ← pass session here
PREFIX       = 'entity-resolution-challenge'

print(f'Bucket : s3://{bucket}/{PREFIX}/')
print(f'Role   : {role[:60]}...')

Bucket : s3://sagemaker-ap-southeast-2-725335003020/entity-resolution-challenge/
Role   : arn:aws:iam::725335003020:role/service-role/AmazonSageMaker-...


In [19]:
from sagemaker import image_uris
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

image_uri = image_uris.retrieve(
    framework='sklearn',
    region=boto_session.region_name,
    version='1.2-1',
    image_scope='training',
)

processor = ScriptProcessor(
    image_uri=image_uri,
    command=['python3'],
    instance_type='ml.t3.xlarge',
    instance_count=1,
    role=role,
    sagemaker_session=session,
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


In [10]:
import os

# find where the notebook is running from
print(os.getcwd())

/home/ec2-user/SageMaker/Amazon_ML_GMNR/notebooks


In [20]:
processor.run(
    code='/home/ec2-user/SageMaker/Amazon_ML_GMNR/src/normalize.py',
    inputs=[
        ProcessingInput(
            source=f's3://{bucket}/{PREFIX}/data/',
            destination='/opt/ml/processing/input',
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{bucket}/{PREFIX}/data/normalized/',
        )
    ],
)

print(f'\nDone. Output at: s3://{bucket}/{PREFIX}/data/normalized/')

INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2026-09-26-01-15-58-523


....................[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
train/train_source1.tsv → train/norm_s1.tsv
train/train_source2.tsv → train/norm_s2.tsv
train/train_source3.tsv → train/norm_s3.tsv
test/test_source1.tsv → test/norm_test_s1.tsv
test/test_source2.tsv → test/norm_test_s2.tsv
test/test_source3.tsv → test/norm_test_s3.tsv


Done. Output at: s3://sagemaker-ap-southeast-2-725335003020/entity-resolution-challenge/data/normalized/


## Verify output

Check that all 6 files were written and spot-check a few rows.

In [21]:
import boto3
import pandas as pd

s3 = boto3.client('s3')

expected_keys = [
    f'{PREFIX}/data/normalized/train/norm_s1.tsv',
    f'{PREFIX}/data/normalized/train/norm_s2.tsv',
    f'{PREFIX}/data/normalized/train/norm_s3.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s1.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s2.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s3.tsv',
]

for key in expected_keys:
    try:
        obj  = s3.head_object(Bucket=bucket, Key=key)
        size = obj['ContentLength'] // 1024
        print(f'  ✓  {key.split("/")[-1]:25s}  {size:>6} KB')
    except Exception:
        print(f'  ✗  MISSING: {key}')

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


  ✓  norm_s1.tsv                393029 KB
  ✓  norm_s2.tsv                880085 KB
  ✓  norm_s3.tsv                929917 KB
  ✓  norm_test_s1.tsv           322644 KB
  ✓  norm_test_s2.tsv           897758 KB
  ✓  norm_test_s3.tsv           917620 KB


In [22]:
# Spot-check norm_s1
norm_s1 = pd.read_csv(
    f's3://{bucket}/{PREFIX}/data/normalized/train/norm_s1.tsv',
    sep='\t', dtype=str,
)

print(f'norm_s1: {len(norm_s1):,} rows, {len(norm_s1.columns)} columns')
print(f'Columns: {list(norm_s1.columns)}\n')
norm_s1.head(5)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


norm_s1: 2,206,821 rows, 11 columns
Columns: ['entity_id', 'country', 'name_norm', 'name_core', 'name_tokens', 'legal_suffix', 'addr_norm', 'addr_tokens', 'pin_zip', 'has_pin', 'landmark']



,entity_id,country,name_norm,name_core,name_tokens,legal_suffix,addr_norm,addr_tokens,pin_zip,has_pin,landmark
0,S1-925783039,US,orelee s barbershop,orelee s barbershop,barbershop orelee s,NaN,1795 westchester dr high point nc,1795 dr high nc point westchester,NaN,False,NaN
1,S1-773889195,US,prime money,prime money,money prime,NaN,17560 ellis rd tahlequah ok,17560 ellis ok rd tahlequah,17560,True,NaN
2,S1-377745466,US,b retail inc,b retail,b inc retail,inc,1712 montebello ave phoenix az,1712 ave az montebello phoenix,NaN,False,NaN
3,S1-133037285,US,christ chapel,christ chapel,chapel christ,NaN,2100 cameron dr unit apt g dundalk md,2100 apt cameron dr dundalk g md unit,NaN,False,NaN
4,S1-755362802,India,prabhav business center,prabhav business,business center prabhav,center,797 lake town blk a kolkata howrah w bengal,797 a bengal blk howrah kolkata lake town w,NaN,False,NaN


In [28]:
import boto3

s3 = boto3.client("s3")

bucket = "sagemaker-ap-southeast-2-725335003020"
prefix = "entity-resolution-challenge/data/normalized/"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=prefix
)

files = response.get("Contents", [])

if files:
    for obj in files:
        print(obj["Key"], f'{obj["Size"] / (1024**2):.2f} MB')
else:
    print("No files found under this S3 prefix.")

entity-resolution-challenge/data/normalized/test/norm_test_s1.tsv 315.08 MB
entity-resolution-challenge/data/normalized/test/norm_test_s2.tsv 876.72 MB
entity-resolution-challenge/data/normalized/test/norm_test_s3.tsv 896.11 MB
entity-resolution-challenge/data/normalized/train/norm_s1.tsv 383.82 MB
entity-resolution-challenge/data/normalized/train/norm_s2.tsv 859.46 MB
entity-resolution-challenge/data/normalized/train/norm_s3.tsv 908.12 MB


In [29]:
import os

DEST = "/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized"
os.makedirs(DEST, exist_ok=True)

for obj in files:
    key = obj["Key"]

    # Skip folder markers
    if key.endswith("/"):
        continue

    filename = os.path.basename(key)
    local_path = os.path.join(DEST, filename)

    s3.download_file(bucket, key, local_path)
    print("Downloaded:", local_path)

print("Files downloaded:", len([
    obj for obj in files if not obj["Key"].endswith("/")
]))

Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_test_s1.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_test_s2.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_test_s3.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s1.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s2.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s3.tsv
Files downloaded: 6


In [5]:
import pandas as pd
def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

norm_s1 = pd.read_csv('/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s1.tsv', sep='\t', dtype=str).fillna('')
norm_s2 = pd.read_csv('/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s2.tsv', sep='\t', dtype=str).fillna('')
norm_s3 = pd.read_csv('/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s3.tsv', sep='\t', dtype=str).fillna('')
train_s1 = pd.read_csv('/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/train/train_source1.tsv', sep='\t', dtype=str).fillna('')
train_s2 = pd.read_csv('/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/train/train_source2.tsv', sep='\t', dtype=str).fillna('')
train_s3 = pd.read_csv('/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/train/train_source3.tsv', sep='\t', dtype=str).fillna('')
ground_truth = pd.read_csv('/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/train_ground_truth.tsv', sep='\t', dtype=str).fillna('')

norm_s1_idx  = norm_s1.set_index('entity_id')
norm_s23_idx = pd.concat([norm_s2, norm_s3]).set_index('entity_id')
raw_s1_idx   = train_s1.set_index('entity_id')
raw_s23_idx  = pd.concat([train_s2, train_s3]).set_index('entity_id')

sample = ground_truth[ground_truth['matched_entity_ids'] != ''].sample(200, random_state=42)

raw_scores, norm_scores = [], []

for _, row in sample.iterrows():
    s1_id   = row['source1_entity_id']
    cand_id = row['matched_entity_ids'].split(',')[0].strip()

    if s1_id not in norm_s1_idx.index or cand_id not in norm_s23_idx.index:
        continue

    raw_a  = str(raw_s1_idx.loc[s1_id,  'business_name']).lower().split()
    raw_b  = str(raw_s23_idx.loc[cand_id, 'business_name']).lower().split()
    norm_a = str(norm_s1_idx.loc[s1_id,  'name_tokens']).split()
    norm_b = str(norm_s23_idx.loc[cand_id, 'name_tokens']).split()

    raw_scores.append(jaccard(raw_a, raw_b))
    norm_scores.append(jaccard(norm_a, norm_b))

mean_raw  = sum(raw_scores)  / len(raw_scores)
mean_norm = sum(norm_scores) / len(norm_scores)
delta     = mean_norm - mean_raw

status = '✅ PASS' if delta >= 0.20 else '❌ FAIL'
print(f'{status}')
print(f'  Mean Jaccard (raw) : {mean_raw:.3f}')
print(f'  Mean Jaccard (norm): {mean_norm:.3f}')
print(f'  Improvement        : {delta:+.3f}  (gate: ≥ +0.20)')

❌ FAIL
  Mean Jaccard (raw) : 0.550
  Mean Jaccard (norm): 0.659
  Improvement        : +0.109  (gate: ≥ +0.20)


In [6]:
sample2 = ground_truth[ground_truth['matched_entity_ids'] != ''].sample(5, random_state=7)

for _, row in sample2.iterrows():
    s1_id   = row['source1_entity_id']
    cand_id = row['matched_entity_ids'].split(',')[0].strip()

    if s1_id not in norm_s1_idx.index or cand_id not in norm_s23_idx.index:
        continue

    print(f'\n── {s1_id}  vs  {cand_id} ──')
    print(f'  name RAW   A : {raw_s1_idx.loc[s1_id, "business_name"]}')
    print(f'  name RAW   B : {raw_s23_idx.loc[cand_id, "business_name"]}')
    print(f'  name NORM  A : {norm_s1_idx.loc[s1_id, "name_norm"]}')
    print(f'  name NORM  B : {norm_s23_idx.loc[cand_id, "name_norm"]}')
    print(f'  addr RAW   A : {raw_s1_idx.loc[s1_id, "business_address"]}')
    print(f'  addr RAW   B : {raw_s23_idx.loc[cand_id, "business_address"]}')
    print(f'  addr NORM  A : {norm_s1_idx.loc[s1_id, "addr_norm"]}')
    print(f'  addr NORM  B : {norm_s23_idx.loc[cand_id, "addr_norm"]}')
    print(f'  pin_zip    A : {norm_s1_idx.loc[s1_id, "pin_zip"]}  |  B : {norm_s23_idx.loc[cand_id, "pin_zip"]}')
    print(f'  landmark   A : {norm_s1_idx.loc[s1_id, "landmark"]}  |  B : {norm_s23_idx.loc[cand_id, "landmark"]}')


── S1-476423513  vs  S2-638379176 ──
  name RAW   A : Continental Resources Pvt Ltd
  name RAW   B : Sri Continental Resources Pvt Ltd
  name NORM  A : continental resources pvt ltd
  name NORM  B : sri continental resources pvt ltd
  addr RAW   A : Plot No C-26 G/F Kh 11/10, 11/11, Shiv Vihar Matiyala, Delhi, West Delhi, Delhi
  addr RAW   B : PLOT NO C-26 G/F KH 11/10, 11/11, SHIV VIHAR MATIYALA, DELHI, Delhi
  addr NORM  A : c 26 g f kh 11 10 11 11 shiv vihar matiyala delhi w delhi delhi
  addr NORM  B : c 26 g f kh 11 10 11 11 shiv vihar matiyala delhi delhi
  pin_zip    A :   |  B : 
  landmark   A :   |  B : 

── S1-211466372  vs  S2-895674642 ──
  name RAW   A : Watkins, Shelden and Mohr Rock Inc
  name RAW   B : WATKINS, SHELDEN AND MR ROCK INC
  name NORM  A : watkins shelden and mohr rock inc
  name NORM  B : watkins shelden and mr rock inc
  addr RAW   A : 9377 Route 152 Highway, Wayne, WV
  addr RAW   B : 9377 ROUTE 152 HIGHWAY, PO BOX 5619, WAYNE, WV
  addr NORM  A : 9377

In [10]:
with open('/home/ec2-user/SageMaker/Amazon_ML_GMNR/src/normalize.py') as f:
    lines = f.readlines()

hits = [
    (i + 1, line.strip())
    for i, line in enumerate(lines)
    if any(kw in line for kw in ["'India'", "'US'", "'France'", '"India"', '"US"', '"France"'])
    and not line.strip().startswith('#')
]

if hits:
    print(f'❌ FAIL — {len(hits)} hits:')
    for lineno, line in hits:
        print(f'  line {lineno}: {line}')
else:
    print('✅ PASS — zero hard-coded country conditionals')

✅ PASS — zero hard-coded country conditionals


In [8]:
import sys
sys.path.insert(0, '/home/ec2-user/SageMaker/Amazon_ML_GMNR')
from src.normalize import extract_legal_suffix

test_cases = [
    ('Acme Private Limited',            'pvt ltd'),
    ('Blue Tech Pvt. Ltd.',             'pvt ltd'),
    ('Global Solutions Pvt Ltd',        'pvt ltd'),
    ('City Traders P Ltd',              'pvt ltd'),
    ('Sun Enterprises P. Ltd.',         'pvt ltd'),
    ('Ravi Exports Limited',            'ltd'),
    ('Metro Supplies Ltd.',             'ltd'),
    ('Alpha Systems Ltd',               'ltd'),
    ('Star Builders Limiteda',          'ltd'),
    ('Prime Goods Limirrad',            'ltd'),
    ('Tech Hub Limitèd',                'ltd'),
    ('Fast Track Límited',              'ltd'),
    ('Green Energy Li',                 'ltd'),
    ('Global Tech LLC',                 'llc'),
    ('City Solutions LLC',              'llc'),
    ('Sunrise Ventures LLP',            'llp'),
    ('Delta Partners LLP',              'llp'),
    ('Ecom Services Elaelapi',          'llp'),
    ('Metro Corp',                      'co'),
    ('City Corporation',                'co'),
    ('National Company',                'co'),
    ('Trade Co',                        'co'),
    ('Retail Com',                      'co'),
    ('Blue Inc',                        'inc'),
    ('Tech Incorporated',               'inc'),
    ('Global Solutions Incorporated',   'inc'),
    ('Alpha Tech Ínc',                  'inc'),
    ('City Pharma Pvt',                 'private'),
    ('Sun Private',                     'private'),
    ('Metro PLC',                       'plc'),
    ('City Bank PLC',                   'plc'),
    ('Café Dupont SAS',                 'sas'),
    ('Boulangerie Martin SARL',         'sarl'),
    ('Immobilier Dupont SCI',           'sci'),
    ('Tech Solutions SASU',             'sasu'),
    ('Import Export EURL',              'eurl'),
    ('Commerce SNC',                    'snc'),
    ('City Center',                     'center'),
    ('Metro Cénter',                    'center'),
    ('Acme Exports',                    None),
    ('Blue Traders',                    None),
    ('Global Foods',                    None),
    ('Sun Industries',                  None),
    ('National Builders',               None),
    ('City Markets',                    None),
    ('Star Logistics',                  None),
    ('Prime Healthcare',                None),
    ('Alpha Textiles',                  None),
    ('Delta Chemicals',                 None),
]

correct = sum(1 for name, expected in test_cases if extract_legal_suffix(name) == expected)
accuracy = correct / len(test_cases)
status   = '✅ PASS' if accuracy >= 0.90 else '❌ FAIL'

print(f'{status}')
print(f'  Accuracy: {correct}/{len(test_cases)} = {accuracy:.0%}  (gate: ≥ 90%)')

# Show any failures
failures = [(name, expected, extract_legal_suffix(name))
            for name, expected in test_cases
            if extract_legal_suffix(name) != expected]
if failures:
    print('\n  Failures:')
    for name, expected, got in failures:
        print(f'    {name!r:45s}  expected={expected}  got={got}')

✅ PASS
  Accuracy: 47/49 = 96%  (gate: ≥ 90%)

  Failures:
    'City Pharma Pvt'                              expected=private  got=pvt
    'Sun Private'                                  expected=private  got=pvt


In [9]:
from src.normalize import STREET_ABBREVS, LEGAL_SUFFIXES

# Paste your noise patterns from Stage 1 EDA here
stage1_patterns = [
    'road', 'street', 'avenue', 'boulevard', 'nagar', 'colony',
    'sector', 'phase', 'junction', 'station', 'market', 'near',
    'opposite', 'limited', 'private', 'corporation', 'company',
    'incorporated', 'pvt ltd', 'private limited', 'llp', 'llc',
    # add more from your noise_patterns dict
]

all_lookups = {**STREET_ABBREVS, **LEGAL_SUFFIXES}
covered   = [p for p in stage1_patterns if p in all_lookups]
uncovered = [p for p in stage1_patterns if p not in all_lookups]
coverage  = len(covered) / len(stage1_patterns)
status    = '✅ PASS' if coverage >= 0.90 else '⚠️  WATCH'

print(f'{status}')
print(f'  Coverage: {len(covered)}/{len(stage1_patterns)} = {coverage:.0%}  (gate: ≥ 90%)')
if uncovered:
    print(f'\n  Not covered — add these to STREET_ABBREVS or LEGAL_SUFFIXES:')
    for p in uncovered:
        print(f'    {p!r}')

✅ PASS
  Coverage: 22/22 = 100%  (gate: ≥ 90%)
